In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The durable loop and its three invariants

Learn how `core.py` operates. Use these cells to operate it. Fill in the `____` blanks. Run each cell. The asserts tell you when your answer is correct. The worked answers are in `../solutions/`, under the same file name.

In [ ]:
import sys; sys.path[:0] = [".."]                # the lab root, from notebooks/ or solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. The run document

Start a run. Then look at the document. This document is *all* that the engine needs to resume the run on any machine, at any later time.

In [ ]:
engine = fresh()
run = engine.start("run-1", "draft", {"topic": "durable agents"})
print(run)
print("queued:", engine.queue.tasks)          # (due, run_id, step, attempt)

## 2. Invariant I2 — a task is only executed if `(step, attempt)` matches

Deliver the first task by hand. Then deliver it *again*, because a queue gives at-least-once delivery. Fill in the value that the second call must return.

In [ ]:
# TODO: fill in: dup, model_calls
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: dup, model_calls")
print(engine.execute("run-1", "draft", 1))
print(engine.execute("run-1", "draft", 1))
assert engine.execute("run-1", "draft", 1) == ____
assert len([c for c in CALLS if c[0] == "model"]) == ____      # the model was called how many times?

## 3. Invariant I1 — checkpoint before enqueue

Read `Engine.execute` in `core.py`. When a step completes with no failure, the engine does two writes at the end of the step. Put the two writes in the correct order. Then say what breaks in the other order.

In [ ]:
# TODO: fill in: first, second
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: first, second")
ORDER = [____, ____]     # "checkpoint" or "enqueue"
WHY = "If we enqueue first and die before the checkpoint, the next task runs against state that was never saved (or is stale forever)."
assert ORDER == ["checkpoint", "enqueue"]

## 4. Invariant I3 — effects are recorded before the checkpoint

`publish` calls `ctx.effect(key, fn)`. Do a simulation of the worst crash: the effect occurred, and then the process crashed before the checkpoint. When the queue delivers the task again, the effect must **not** run again.

In [ ]:
# TODO: fill in: seconds
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: seconds")
engine = fresh()
engine.start("run-1", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
engine.resume("run-1", "approval:run-1", {"decision": "approve"})

real_publish, died = engine.steps["publish"], []
def publish_then_die(ctx):
    out = real_publish(ctx)                 # cms_publish happened here
    if not died:
        died.append(1); raise Crash("died before the checkpoint")
    return out
engine.steps["publish"] = publish_then_die

print(drain(engine, engine.queue, now))
clock[0] += ____                      # wait for the dead worker's lease to expire
print(drain(engine, engine.queue, now))
publishes = [c for c in CALLS if c[0] == "publish"]
print("publish calls:", len(publishes), "| result:", engine.store.get("run-1")["result"])
assert len(publishes) == 1 and engine.store.get("run-1")["status"] == "SUCCEEDED"

## 5. Your turn: add an idempotent effect

Write a `bill` step. The step charges a card **at most once**, even if the engine retries the step. Then the step continues to `done_step`.

In [ ]:
# TODO: fill in: key
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: key")
CHARGES = []
def bill(ctx):
    rec = ctx.effect(____, lambda: CHARGES.append(ctx.input["amount"]) or {"charge_id": "ch_1"})
    ctx.state["charge_id"] = rec["charge_id"]
    return ("next", "done_step")
def done_step(ctx):
    return ("done", ctx.state["charge_id"])

engine = fresh({"bill": bill, "done_step": done_step})
engine.start("inv-1", "bill", {"amount": 42})
engine.execute("inv-1", "bill", 1)
engine.execute("inv-1", "bill", 1)            # duplicate delivery
drain(engine, engine.queue, now)
assert CHARGES == [42] and engine.store.get("inv-1")["result"] == "ch_1"
print("charged exactly once:", CHARGES)